In [3]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
print("Python:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("현재 위치:", Path.cwd())


Python: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
pandas: 3.0.3
numpy: 2.5.0
현재 위치: /home/soldesk/ai-hybrid-lab-kim-juil/notebooks


In [4]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
CLEAN_DIR = BASE_DIR / "data" / "clean"
FEATURE_DIR = BASE_DIR / "data" / "feature"
OUTPUT_DIR = BASE_DIR / "outputs" / "week2"
REPORT_DIR = BASE_DIR / "reports" / "week2"
for path in [RAW_DIR, CLEAN_DIR, FEATURE_DIR, OUTPUT_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("CLEAN_DIR:", CLEAN_DIR)
print("FEATURE_DIR:", FEATURE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("REPORT_DIR:", REPORT_DIR)


BASE_DIR: /home/soldesk/ai-hybrid-lab-kim-juil
RAW_DIR: /home/soldesk/ai-hybrid-lab-kim-juil/data/raw
CLEAN_DIR: /home/soldesk/ai-hybrid-lab-kim-juil/data/clean
FEATURE_DIR: /home/soldesk/ai-hybrid-lab-kim-juil/data/feature
OUTPUT_DIR: /home/soldesk/ai-hybrid-lab-kim-juil/outputs/week2
REPORT_DIR: /home/soldesk/ai-hybrid-lab-kim-juil/reports/week2


In [5]:
raw_file = RAW_DIR / "customers_raw.csv"
print("raw_file:", raw_file)
print("exists:", raw_file.exists())
if raw_file.exists():
    print("size:", raw_file.stat().st_size)
df_raw = pd.read_csv(raw_file)
print(df_raw.shape)
df_raw.head()


raw_file: /home/soldesk/ai-hybrid-lab-kim-juil/data/raw/customers_raw.csv
exists: True
size: 375
(10, 8)


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn
0,C001,34.0,F,59000.0,monthly,120,1,0
1,C002,45.0,M,79000.0,yearly,380,3,1
2,C003,NaN,F,NaN,monthly,45,0,0
3,C004,52.0,M,99000.0,yearly,800,5,1
4,C005,41.0,F,69000.0,NaN,220,2,0


In [6]:
missing_examples = pd.DataFrame({
    "label": ["NaN", "None", "empty_string", "unknown", "N/A", "-"],
    "value": [np.nan, None, "", "unknown", "N/A", "-"]
})
missing_examples["isnull_result"] = missing_examples["value"].isnull()
missing_examples

,label,value,isnull_result
0,NaN,NaN,True
1,None,NaN,True
2,empty_string,,False
3,unknown,unknown,False
4,N/A,N/A,False
5,-,-,False


In [7]:
missing_count = df_raw.isnull().sum()
missing_count

customer_id      0
age              1
gender           1
monthly_fee      1
contract_type    2
usage_days       0
support_calls    0
churn            0
dtype: int64

In [8]:
missing_ratio = df_raw.isnull().mean() * 100
missing_ratio

customer_id       0.0
age              10.0
gender           10.0
monthly_fee      10.0
contract_type    20.0
usage_days        0.0
support_calls     0.0
churn             0.0
dtype: float64

In [9]:
missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_ratio_percent": missing_ratio,
    "dtype": df_raw.dtypes.astype(str)
})
missing_report = missing_report.sort_values(
    "missing_count",
    ascending=False
)
missing_report

,missing_count,missing_ratio_percent,dtype
contract_type,2,20.0,str
age,1,10.0,float64
monthly_fee,1,10.0,float64
gender,1,10.0,str
customer_id,0,0.0,str
usage_days,0,0.0,int64
support_calls,0,0.0,int64
churn,0,0.0,int64


In [10]:
missing_report_with_missing = missing_report[
    missing_report["missing_count"] > 0
]
missing_report_with_missing

,missing_count,missing_ratio_percent,dtype
contract_type,2,20.0,str
age,1,10.0,float64
monthly_fee,1,10.0,float64
gender,1,10.0,str


In [11]:
rows_with_missing = df_raw[df_raw.isnull().any(axis=1)]
rows_with_missing

,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn
2,C003,NaN,F,NaN,monthly,45,0,0
4,C005,41.0,F,69000.0,NaN,220,2,0
5,C005,41.0,F,69000.0,NaN,220,2,0
8,C008,23.0,NaN,45000.0,monthly,15,0,0


In [12]:
df_raw["gender"].value_counts(dropna=False)

gender
F      5
M      4
NaN    1
Name: count, dtype: int64

In [13]:
df_raw["contract_type"].value_counts(dropna=False)

contract_type
monthly    4
yearly     4
NaN        2
Name: count, dtype: int64

In [14]:
categorical_cols = ["gender", "contract_type"]
for col in categorical_cols:
    print("=" * 60)
    print("컬럼:", col)
    print(df_raw[col].value_counts(dropna=False))


컬럼: gender
gender
F      5
M      4
NaN    1
Name: count, dtype: int64
컬럼: contract_type
contract_type
monthly    4
yearly     4
NaN        2
Name: count, dtype: int64


In [15]:
action_map = {
    "age": "median_imputation",
    "monthly_fee": "median_imputation",
    "gender": "unknown_imputation",
    "contract_type": "unknown_imputation"
}
missing_report["suggested_action"] = missing_report.index.map(
    lambda col: action_map.get(col, "no_action")
)
missing_report

,missing_count,missing_ratio_percent,dtype,suggested_action
contract_type,2,20.0,str,unknown_imputation
age,1,10.0,float64,median_imputation
monthly_fee,1,10.0,float64,median_imputation
gender,1,10.0,str,unknown_imputation
customer_id,0,0.0,str,no_action
usage_days,0,0.0,int64,no_action
support_calls,0,0.0,int64,no_action
churn,0,0.0,int64,no_action


In [16]:
import json
from pathlib import Path

# 현재 Notebook 실행 기준 디렉터리 확인
CURRENT_DIR = Path.cwd()

# Notebook이 project-root/notebooks 에서 실행되는 경우
# PROJECT_ROOT를 한 단계 위로 설정
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# 프로젝트 루트 하위 outputs/week2 디렉터리
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "week2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# reason 컬럼이 없으면 자동 생성
if "reason" not in missing_report.columns:
    def make_reason(row):
        missing_count_value = int(row["missing_count"])

        if missing_count_value == 0:
            return "결측치가 없으므로 별도 처리가 필요하지 않음"

        action = str(row.get("suggested_action", ""))

        if "median" in action or "중앙값" in action:
            return "수치형 컬럼이며 이상치 영향을 줄이기 위해 중앙값 대체 후보로 판단"
        elif "mean" in action or "평균" in action:
            return "수치형 컬럼이며 값 분포가 비교적 안정적이면 평균 대체 가능"
        elif "unknown" in action or "최빈값" in action:
            return "범주형 컬럼이며 비어 있는 값을 별도 범주 또는 대표값으로 처리 가능"
        elif "drop" in action or "삭제" in action:
            return "결측 비율이 높거나 분석에 적합하지 않아 삭제 후보로 판단"
        else:
            return "결측치가 존재하므로 데이터 의미를 확인한 뒤 처리 전략 결정 필요"

    missing_report["reason"] = missing_report.apply(make_reason, axis=1)

missing_analysis_summary = {
    "author": "student",
    "input_raw_path": str(raw_file),
    "row_count": int(df_raw.shape[0]),
    "column_count": int(df_raw.shape[1]),
    "missing_columns": list(
        missing_report[missing_report["missing_count"] > 0].index
    ),
    "missing_count": {
        col: int(value)
        for col, value in missing_count.items()
    },
    "missing_ratio_percent": {
        col: float(value)
        for col, value in missing_ratio.items()
    },
    "suggested_action": {
        col: str(missing_report.loc[col, "suggested_action"])
        for col in missing_report.index
    },
    "reason": {
        col: str(missing_report.loc[col, "reason"])
        for col in missing_report.index
    },
    "rows_with_missing_count": int(len(rows_with_missing)),
    "g4dn_used": False,
    "openshift_deploy_used": False,
    "note": "Day 2 2교시는 결측치를 처리하지 않고 개수, 비율, 실제 행, 처리 후보를 분석한 시간"
}

missing_output_path = OUTPUT_DIR / "day2_missing_analysis_summary.json"

with open(missing_output_path, "w", encoding="utf-8") as f:
    json.dump(missing_analysis_summary, f, ensure_ascii=False, indent=2)

print("current dir:", CURRENT_DIR)
print("project root:", PROJECT_ROOT)
print("saved:", missing_output_path)
print("exists:", missing_output_path.exists())

missing_analysis_summary

current dir: /home/soldesk/ai-hybrid-lab-kim-juil/notebooks
project root: /home/soldesk/ai-hybrid-lab-kim-juil
saved: /home/soldesk/ai-hybrid-lab-kim-juil/outputs/week2/day2_missing_analysis_summary.json
exists: True


{'author': 'student',
 'input_raw_path': '/home/soldesk/ai-hybrid-lab-kim-juil/data/raw/customers_raw.csv',
 'row_count': 10,
 'column_count': 8,
 'missing_columns': ['contract_type', 'age', 'monthly_fee', 'gender'],
 'missing_count': {'customer_id': 0,
  'age': 1,
  'gender': 1,
  'monthly_fee': 1,
  'contract_type': 2,
  'usage_days': 0,
  'support_calls': 0,
  'churn': 0},
 'missing_ratio_percent': {'customer_id': 0.0,
  'age': 10.0,
  'gender': 10.0,
  'monthly_fee': 10.0,
  'contract_type': 20.0,
  'usage_days': 0.0,
  'support_calls': 0.0,
  'churn': 0.0},
 'suggested_action': {'contract_type': 'unknown_imputation',
  'age': 'median_imputation',
  'monthly_fee': 'median_imputation',
  'gender': 'unknown_imputation',
  'customer_id': 'no_action',
  'usage_days': 'no_action',
  'support_calls': 'no_action',
  'churn': 'no_action'},
 'reason': {'contract_type': '범주형 컬럼이며 비어 있는 값을 별도 범주 또는 대표값으로 처리 가능',
  'age': '수치형 컬럼이며 이상치 영향을 줄이기 위해 중앙값 대체 후보로 판단',
  'monthly_fee': '수치형 컬럼이며 이상치 

In [17]:
print("df_raw 크기:", df_raw.shape)

df_raw 크기: (10, 8)


In [18]:
df_raw.head()

,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn
0,C001,34.0,F,59000.0,monthly,120,1,0
1,C002,45.0,M,79000.0,yearly,380,3,1
2,C003,NaN,F,NaN,monthly,45,0,0
3,C004,52.0,M,99000.0,yearly,800,5,1
4,C005,41.0,F,69000.0,NaN,220,2,0


In [19]:
df_raw.isnull().sum()

customer_id      0
age              1
gender           1
monthly_fee      1
contract_type    2
usage_days       0
support_calls    0
churn            0
dtype: int64

In [20]:
df_original = df_raw.copy()
print("원본 행 수:", len(df_original))
print("원본 결측치 총합:", int(df_original.isnull().sum().sum()))


원본 행 수: 10
원본 결측치 총합: 5


In [21]:
df_dropna = df_raw.dropna().copy()
print("원본 행 수:", len(df_raw))
print("dropna 후 행 수:", len(df_dropna))
print("제거된 행 수:", len(df_raw) - len(df_dropna))


원본 행 수: 10
dropna 후 행 수: 6
제거된 행 수: 4


In [22]:
removed_rows = len(df_raw) - len(df_dropna)
removed_ratio = removed_rows / len(df_raw) * 100
print("제거된 행 비율:", removed_ratio)

제거된 행 비율: 40.0


In [23]:
raw_churn_dist = df_raw["churn"].value_counts(dropna=False).sort_index()
dropna_churn_dist = df_dropna["churn"].value_counts(dropna=False).sort_index()
print("원본 churn 분포")
print(raw_churn_dist)
print("dropna 후 churn 분포")
print(dropna_churn_dist)


원본 churn 분포
churn
0    6
1    4
Name: count, dtype: int64
dropna 후 churn 분포
churn
0    2
1    4
Name: count, dtype: int64


In [24]:
raw_churn_ratio = df_raw["churn"].value_counts(normalize=True, dropna=False).sort_index() * 100
dropna_churn_ratio = df_dropna["churn"].value_counts(normalize=True, dropna=False).sort_index() * 100
churn_compare = pd.DataFrame({
    "raw_percent": raw_churn_ratio,
    "dropna_percent": dropna_churn_ratio
})
churn_compare

,raw_percent,dropna_percent
churn,,
0,60.0,33.333333
1,40.0,66.666667


In [25]:
df_imputed = df_raw.copy()
print("대체 전 행 수:", len(df_imputed))
print("대체 전 결측치 총합:", int(df_imputed.isnull().sum().sum()))


대체 전 행 수: 10
대체 전 결측치 총합: 5


In [26]:
missing_target_cols = ["age", "monthly_fee", "gender", "contract_type"]
for col in missing_target_cols:
    df_imputed[f"{col}_was_missing"] = df_imputed[col].isnull().astype(int)
df_imputed[
    [
        "customer_id",
        "age_was_missing",
        "monthly_fee_was_missing",
        "gender_was_missing",
        "contract_type_was_missing"
    ]
].head()


,customer_id,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
0,C001,0,0,0,0
1,C002,0,0,0,0
2,C003,1,1,0,0
3,C004,0,0,0,0
4,C005,0,0,0,1


In [27]:
numeric_missing_cols = ["age", "monthly_fee"]
for col in numeric_missing_cols:
    mean_value = df_imputed[col].mean()
    median_value = df_imputed[col].median()
    print("=" * 60)
    print("컬럼:", col)
    print("평균:", mean_value)
    print("중앙값:", median_value)


컬럼: age
평균: 43.77777777777778
중앙값: 41.0
컬럼: monthly_fee
평균: 80222.22222222222
중앙값: 72000.0


In [28]:
numeric_missing_cols = ["age", "monthly_fee"]
for col in numeric_missing_cols:
    mean_value = df_imputed[col].mean()
    median_value = df_imputed[col].median()
    print("=" * 60)
    print("컬럼:", col)
    print("평균:", mean_value)
    print("중앙값:", median_value)


컬럼: age
평균: 43.77777777777778
중앙값: 41.0
컬럼: monthly_fee
평균: 80222.22222222222
중앙값: 72000.0


In [29]:
age_median = df_imputed["age"].median()
monthly_fee_median = df_imputed["monthly_fee"].median()
df_imputed["age"] = df_imputed["age"].fillna(age_median)
df_imputed["monthly_fee"] = df_imputed["monthly_fee"].fillna(monthly_fee_median)
print("age 대체값:", age_median)
print("monthly_fee 대체값:", monthly_fee_median)


age 대체값: 41.0
monthly_fee 대체값: 72000.0


In [30]:
df_imputed["gender"] = df_imputed["gender"].fillna("unknown")
df_imputed["contract_type"] = df_imputed["contract_type"].fillna("unknown")


In [31]:
df_imputed.isnull().sum()

customer_id                  0
age                          0
gender                       0
monthly_fee                  0
contract_type                0
usage_days                   0
support_calls                0
churn                        0
age_was_missing              0
monthly_fee_was_missing      0
gender_was_missing           0
contract_type_was_missing    0
dtype: int64

In [32]:
comparison_rows = pd.DataFrame({
    "dataset": ["raw", "dropna", "imputed"],
    "row_count": [
        len(df_raw),
        len(df_dropna),
        len(df_imputed)
    ],
    "missing_total": [
        int(df_raw.isnull().sum().sum()),
        int(df_dropna.isnull().sum().sum()),
        int(df_imputed.isnull().sum().sum())
    ]
})
comparison_rows

,dataset,row_count,missing_total
0,raw,10,5
1,dropna,6,0
2,imputed,10,0


In [33]:
numeric_compare = pd.DataFrame({
    "raw_mean": df_raw[["age", "monthly_fee"]].mean(),
    "raw_median": df_raw[["age", "monthly_fee"]].median(),
    "imputed_mean": df_imputed[["age", "monthly_fee"]].mean(),
    "imputed_median": df_imputed[["age", "monthly_fee"]].median()
})
numeric_compare

,raw_mean,raw_median,imputed_mean,imputed_median
age,43.777778,41.0,43.5,41.0
monthly_fee,80222.222222,72000.0,79400.0,72000.0


In [34]:
print("원본 gender 분포")
print(df_raw["gender"].value_counts(dropna=False))
print("\n대체 후 gender 분포")
print(df_imputed["gender"].value_counts(dropna=False))
print("\n원본 contract_type 분포")
print(df_raw["contract_type"].value_counts(dropna=False))
print("\n대체 후 contract_type 분포")
print(df_imputed["contract_type"].value_counts(dropna=False))


원본 gender 분포
gender
F      5
M      4
NaN    1
Name: count, dtype: int64

대체 후 gender 분포
gender
F          5
M          4
unknown    1
Name: count, dtype: int64

원본 contract_type 분포
contract_type
monthly    4
yearly     4
NaN        2
Name: count, dtype: int64

대체 후 contract_type 분포
contract_type
monthly    4
yearly     4
unknown    2
Name: count, dtype: int64


In [35]:
df_imputed["gender"] = df_imputed["gender"].fillna("unknown")
df_imputed["contract_type"] = df_imputed["contract_type"].fillna("unknown")


In [36]:
df_imputed.isnull().sum()


customer_id                  0
age                          0
gender                       0
monthly_fee                  0
contract_type                0
usage_days                   0
support_calls                0
churn                        0
age_was_missing              0
monthly_fee_was_missing      0
gender_was_missing           0
contract_type_was_missing    0
dtype: int64

In [37]:
missing_imputed_candidate_path = CLEAN_DIR / "customers_missing_imputed_candidate_v1.csv"
df_imputed.to_csv(missing_imputed_candidate_path, index=False)
print("saved:", missing_imputed_candidate_path)
print("exists:", missing_imputed_candidate_path.exists())


saved: /home/soldesk/ai-hybrid-lab-kim-juil/data/clean/customers_missing_imputed_candidate_v1.csv
exists: True


In [38]:
import json
from pathlib import Path
missing_treatment_summary = {
    "author": "student",
    "input_raw_path": str(raw_file),
    "raw_rows": int(len(df_raw)),
    "raw_missing_total": int(df_raw.isnull().sum().sum()),
    "dropna_rows": int(len(df_dropna)),
    "dropna_removed_rows": int(len(df_raw) - len(df_dropna)),
    "dropna_removed_ratio_percent": float((len(df_raw) - len(df_dropna)) / len(df_raw) * 100),
    "imputed_rows": int(len(df_imputed)),
    "imputed_missing_total": int(df_imputed.isnull().sum().sum()),
    "numeric_strategy": {
        "age": {
            "strategy": "median",
            "value": float(age_median)
        },
        "monthly_fee": {
            "strategy": "median",
            "value": float(monthly_fee_median)
        }
    },
    "categorical_strategy": {
        "gender": {
            "strategy": "unknown"
        },
        "contract_type": {
            "strategy": "unknown"
        }
    },
    "missing_indicator_columns": [
        "age_was_missing",
        "monthly_fee_was_missing",
        "gender_was_missing",
        "contract_type_was_missing"
    ],
    "selected_strategy_for_day2": "imputation",
    "reason": "행 수가 적어 dropna로 삭제하면 데이터 손실이 크므로, 수치형은 중앙값, 범주형은 unknown으로 대체하는 방식을 Day 2 기본 후보로 선택",
    "middle_clean_candidate_path": str(missing_imputed_candidate_path),
    "g4dn_used": False,
    "openshift_deploy_used": False,
    "note": "Day 2 3교시는 결측치 제거와 대체를 비교하고 대체 후보를 기록한 시간"
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
missing_treatment_output_path = OUTPUT_DIR / "day2_missing_treatment_comparison.json"
with open(missing_treatment_output_path, "w", encoding="utf-8") as f:
    json.dump(missing_treatment_summary, f, ensure_ascii=False, indent=2)
print("saved:", missing_treatment_output_path)
missing_treatment_summary


saved: /home/soldesk/ai-hybrid-lab-kim-juil/outputs/week2/day2_missing_treatment_comparison.json


{'author': 'student',
 'input_raw_path': '/home/soldesk/ai-hybrid-lab-kim-juil/data/raw/customers_raw.csv',
 'raw_rows': 10,
 'raw_missing_total': 5,
 'dropna_rows': 6,
 'dropna_removed_rows': 4,
 'dropna_removed_ratio_percent': 40.0,
 'imputed_rows': 10,
 'imputed_missing_total': 0,
 'numeric_strategy': {'age': {'strategy': 'median', 'value': 41.0},
  'monthly_fee': {'strategy': 'median', 'value': 72000.0}},
 'categorical_strategy': {'gender': {'strategy': 'unknown'},
  'contract_type': {'strategy': 'unknown'}},
 'missing_indicator_columns': ['age_was_missing',
  'monthly_fee_was_missing',
  'gender_was_missing',
  'contract_type_was_missing'],
 'selected_strategy_for_day2': 'imputation',
 'reason': '행 수가 적어 dropna로 삭제하면 데이터 손실이 크므로, 수치형은 중앙값, 범주형은 unknown으로 대체하는 방식을 Day 2 기본 후보로 선택',
 'middle_clean_candidate_path': '/home/soldesk/ai-hybrid-lab-kim-juil/data/clean/customers_missing_imputed_candidate_v1.csv',
 'g4dn_used': False,
 'openshift_deploy_used': False,
 'note': 'Day 2 3교시는 결측

In [39]:
print("df_raw:", df_raw.shape)
print("df_imputed:", df_imputed.shape)
df_imputed.head()

df_raw: (10, 8)
df_imputed: (10, 12)


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
0,C001,34.0,F,59000.0,monthly,120,1,0,0,0,0,0
1,C002,45.0,M,79000.0,yearly,380,3,1,0,0,0,0
2,C003,41.0,F,72000.0,monthly,45,0,0,1,1,0,0
3,C004,52.0,M,99000.0,yearly,800,5,1,0,0,0,0
4,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1


In [40]:
df_quality = df_imputed.copy()
print("df_quality 행/열:", df_quality.shape)
df_quality.head()


df_quality 행/열: (10, 12)


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
0,C001,34.0,F,59000.0,monthly,120,1,0,0,0,0,0
1,C002,45.0,M,79000.0,yearly,380,3,1,0,0,0,0
2,C003,41.0,F,72000.0,monthly,45,0,0,1,1,0,0
3,C004,52.0,M,99000.0,yearly,800,5,1,0,0,0,0
4,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1


In [41]:
full_duplicate_count = df_quality.duplicated().sum()
print("완전 중복 수:", full_duplicate_count)

완전 중복 수: 1


In [42]:
df_quality[df_quality.duplicated(keep=False)]

,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
4,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1
5,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1


In [43]:
df_quality[df_quality.duplicated(keep="first")]

,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
5,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1


In [44]:
df_quality[df_quality.duplicated(keep="last")]

,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
4,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1


In [45]:
id_duplicate_count = df_quality.duplicated(subset=["customer_id"]).sum()
print("customer_id 기준 중복 수:", id_duplicate_count)

customer_id 기준 중복 수: 1


In [46]:
df_quality[df_quality.duplicated(subset=["customer_id"], keep=False)].sort_values("customer_id")


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
4,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1
5,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1


In [47]:
full_duplicate_count = df_quality.duplicated().sum()
print("완전 중복 수:", full_duplicate_count)

완전 중복 수: 1


In [48]:
df_quality[df_quality.duplicated(keep=False)]

,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
4,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1
5,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1


In [49]:
df_quality[df_quality.duplicated(subset=["customer_id"], keep=False)].sort_values("customer_id")

,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
4,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1
5,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1


In [50]:
df_dup_demo = pd.concat(
    [
        df_quality,
        df_quality.iloc[[0]],
        df_quality.iloc[[0]].assign(monthly_fee=df_quality.iloc[0]["monthly_fee"] + 10000)
    ],
    ignore_index=True
)
df_dup_demo.tail()

,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
7,C007,58.0,F,110000.0,yearly,1000,6,1,0,0,0,0
8,C008,23.0,unknown,45000.0,monthly,15,0,0,0,0,1,0
9,C009,63.0,M,120000.0,yearly,1100,7,1,0,0,0,0
10,C001,34.0,F,59000.0,monthly,120,1,0,0,0,0,0
11,C001,34.0,F,69000.0,monthly,120,1,0,0,0,0,0


In [51]:
print("완전 중복 수:", df_dup_demo.duplicated().sum())
print("customer_id 기준 중복 수:", df_dup_demo.duplicated(subset=["customer_id"]).sum())


완전 중복 수: 2
customer_id 기준 중복 수: 3


In [52]:
df_dup_demo

,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
0,C001,34.0,F,59000.0,monthly,120,1,0,0,0,0,0
1,C002,45.0,M,79000.0,yearly,380,3,1,0,0,0,0
2,C003,41.0,F,72000.0,monthly,45,0,0,1,1,0,0
3,C004,52.0,M,99000.0,yearly,800,5,1,0,0,0,0
4,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1
5,C005,41.0,F,69000.0,unknown,220,2,0,0,0,0,1
6,C006,37.0,M,72000.0,monthly,90,4,0,0,0,0,0
7,C007,58.0,F,110000.0,yearly,1000,6,1,0,0,0,0
8,C008,23.0,unknown,45000.0,monthly,15,0,0,0,0,1,0
9,C009,63.0,M,120000.0,yearly,1100,7,1,0,0,0,0


In [53]:
before_rows = len(df_quality)
df_quality = df_quality.drop_duplicates().copy()
after_rows = len(df_quality)
print("중복 제거 전 행 수:", before_rows)
print("중복 제거 후 행 수:", after_rows)
print("제거된 완전 중복 수:", before_rows - after_rows)


중복 제거 전 행 수: 10
중복 제거 후 행 수: 9
제거된 완전 중복 수: 1


In [54]:
df_quality.dtypes
df_quality.info()

<class 'pandas.DataFrame'>
Index: 9 entries, 0 to 9
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer_id                9 non-null      str    
 1   age                        9 non-null      float64
 2   gender                     9 non-null      str    
 3   monthly_fee                9 non-null      float64
 4   contract_type              9 non-null      str    
 5   usage_days                 9 non-null      int64  
 6   support_calls              9 non-null      int64  
 7   churn                      9 non-null      int64  
 8   age_was_missing            9 non-null      int64  
 9   monthly_fee_was_missing    9 non-null      int64  
 10  gender_was_missing         9 non-null      int64  
 11  contract_type_was_missing  9 non-null      int64  
dtypes: float64(2), int64(7), str(3)
memory usage: 1.0 KB


In [55]:
id_col = "customer_id"
target_col = "churn"
numeric_cols = [
    "age",
    "monthly_fee",
    "usage_days",
    "support_calls"
]
categorical_cols = [
    "gender",
    "contract_type"
]
missing_indicator_cols = [
    "age_was_missing",
    "monthly_fee_was_missing",
    "gender_was_missing",
    "contract_type_was_missing"
]
print("ID 컬럼:", id_col)
print("Target 컬럼:", target_col)
print("수치형 컬럼:", numeric_cols)
print("범주형 컬럼:", categorical_cols)
print("결측 여부 컬럼:", missing_indicator_cols)


ID 컬럼: customer_id
Target 컬럼: churn
수치형 컬럼: ['age', 'monthly_fee', 'usage_days', 'support_calls']
범주형 컬럼: ['gender', 'contract_type']
결측 여부 컬럼: ['age_was_missing', 'monthly_fee_was_missing', 'gender_was_missing', 'contract_type_was_missing']


In [56]:
df_quality["age"] = df_quality["age"].astype(int)
df_quality["monthly_fee"] = df_quality["monthly_fee"].astype(int)
for col in missing_indicator_cols:
    df_quality[col] = df_quality[col].astype(int)
df_quality["churn"] = df_quality["churn"].astype(int)
df_quality.dtypes

customer_id                    str
age                          int64
gender                         str
monthly_fee                  int64
contract_type                  str
usage_days                   int64
support_calls                int64
churn                        int64
age_was_missing              int64
monthly_fee_was_missing      int64
gender_was_missing           int64
contract_type_was_missing    int64
dtype: object

In [57]:
string_quality_examples = pd.DataFrame({
    "raw_value": [
        " F",
        "f ",
        "Female",
        " female ",
        "M",
        " male",
        "UNKNOWN ",
        "",
        None
    ]
})
string_quality_examples

,raw_value
0,F
1,f
2,Female
3,female
4,M
5,male
6,UNKNOWN
7,
8,NaN


In [58]:
def normalize_text(value):
    if pd.isna(value):
        return "unknown"

    value = str(value).strip().lower()

    if value == "":
        return "unknown"

    if value in ["nan", "none", "null", "n/a", "-"]:
        return "unknown"

    return value

In [59]:
string_quality_examples["normalized_value"] = string_quality_examples["raw_value"].apply(normalize_text)
string_quality_examples

,raw_value,normalized_value
0,F,f
1,f,f
2,Female,female
3,female,female
4,M,m
5,male,male
6,UNKNOWN,unknown
7,,unknown
8,NaN,unknown


In [60]:
print("정리 전 gender")
print(df_quality["gender"].value_counts(dropna=False))
print("\n정리 전 contract_type")
print(df_quality["contract_type"].value_counts(dropna=False))


정리 전 gender
gender
F          4
M          4
unknown    1
Name: count, dtype: int64

정리 전 contract_type
contract_type
monthly    4
yearly     4
unknown    1
Name: count, dtype: int64


In [61]:
df_quality["gender_clean"] = df_quality["gender"].apply(normalize_text)
df_quality["contract_type_clean"] = df_quality["contract_type"].apply(normalize_text)


In [62]:
print("정리 후 gender_clean")
print(df_quality["gender_clean"].value_counts(dropna=False))
print("\n정리 후 contract_type_clean")
print(df_quality["contract_type_clean"].value_counts(dropna=False))


정리 후 gender_clean
gender_clean
f          4
m          4
unknown    1
Name: count, dtype: int64

정리 후 contract_type_clean
contract_type_clean
monthly    4
yearly     4
unknown    1
Name: count, dtype: int64


In [63]:
gender_map = {
    "m": "male",
    "male": "male",
    "f": "female",
    "female": "female",
    "unknown": "unknown"
}
df_quality["gender_std"] = df_quality["gender_clean"].map(gender_map).fillna("unknown")
df_quality[["gender", "gender_clean", "gender_std"]].head()

,gender,gender_clean,gender_std
0,F,f,female
1,M,m,male
2,F,f,female
3,M,m,male
4,F,f,female


In [64]:
df_quality["gender_std"].value_counts(dropna=False)

gender_std
female     4
male       4
unknown    1
Name: count, dtype: int64

In [65]:
contract_map = {
    "monthly": "month-to-month",
    "month-to-month": "month-to-month",
    "m2m": "month-to-month",
    "yearly": "1-year",
    "1-year": "1-year",
    "one-year": "1-year",
    "2-year": "2-year",
    "two-year": "2-year",
    "unknown": "unknown"
}
df_quality["contract_type_std"] = (
    df_quality["contract_type_clean"]
    .map(contract_map)
    .fillna("unknown")
)
df_quality[["contract_type", "contract_type_clean", "contract_type_std"]].head()


,contract_type,contract_type_clean,contract_type_std
0,monthly,monthly,month-to-month
1,yearly,yearly,1-year
2,monthly,monthly,month-to-month
3,yearly,yearly,1-year
4,unknown,unknown,unknown


In [66]:
df_quality["contract_type_std"].value_counts(dropna=False)

contract_type_std
month-to-month    4
1-year            4
unknown           1
Name: count, dtype: int64

In [67]:
allowed_gender_values = {"male", "female", "unknown"}
allowed_contract_values = {
    "month-to-month",
    "1-year",
    "2-year",
    "unknown"
}


In [68]:
gender_invalid_values = set(df_quality["gender_std"].unique()) - allowed_gender_values
contract_invalid_values = set(df_quality["contract_type_std"].unique()) - allowed_contract_values
print("gender 허용되지 않은 값:", gender_invalid_values)
print("contract_type 허용되지 않은 값:", contract_invalid_values)


gender 허용되지 않은 값: set()
contract_type 허용되지 않은 값: set()


In [69]:
df_quality["gender"] = df_quality["gender_std"]
df_quality["contract_type"] = df_quality["contract_type_std"]
df_quality = df_quality.drop(
    columns=[
        "gender_clean",
        "gender_std",
        "contract_type_clean",
        "contract_type_std"
    ]
)
df_quality.head()

,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
0,C001,34,female,59000,month-to-month,120,1,0,0,0,0,0
1,C002,45,male,79000,1-year,380,3,1,0,0,0,0
2,C003,41,female,72000,month-to-month,45,0,0,1,1,0,0
3,C004,52,male,99000,1-year,800,5,1,0,0,0,0
4,C005,41,female,69000,unknown,220,2,0,0,0,0,1


In [70]:
df_quality

,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
0,C001,34,female,59000,month-to-month,120,1,0,0,0,0,0
1,C002,45,male,79000,1-year,380,3,1,0,0,0,0
2,C003,41,female,72000,month-to-month,45,0,0,1,1,0,0
3,C004,52,male,99000,1-year,800,5,1,0,0,0,0
4,C005,41,female,69000,unknown,220,2,0,0,0,0,1
6,C006,37,male,72000,month-to-month,90,4,0,0,0,0,0
7,C007,58,female,110000,1-year,1000,6,1,0,0,0,0
8,C008,23,unknown,45000,month-to-month,15,0,0,0,0,1,0
9,C009,63,male,120000,1-year,1100,7,1,0,0,0,0


In [71]:
quality_check_result = {
    "row_count": int(len(df_quality)),
    "full_duplicate_count": int(df_quality.duplicated().sum()),
    "customer_id_duplicate_count": int(df_quality.duplicated(subset=["customer_id"]).sum()),
    "missing_total": int(df_quality.isnull().sum().sum()),
    "gender_values": sorted(df_quality["gender"].unique().tolist()),
    "contract_type_values": sorted(df_quality["contract_type"].unique().tolist())
}
quality_check_result

{'row_count': 9,
 'full_duplicate_count': 0,
 'customer_id_duplicate_count': 0,
 'missing_total': 0,
 'gender_values': ['female', 'male', 'unknown'],
 'contract_type_values': ['1-year', 'month-to-month', 'unknown']}

In [72]:
import json
from pathlib import Path
quality_summary = {
    "author": "student",
    "input_dataframe": "df_imputed",
    "working_dataframe": "df_quality",
    "row_count": int(len(df_quality)),
    "full_duplicate_count": int(df_quality.duplicated().sum()),
    "customer_id_duplicate_count": int(df_quality.duplicated(subset=["customer_id"]).sum()),
    "dtypes": {
        col: str(dtype)
        for col, dtype in df_quality.dtypes.items()
    },
    "numeric_columns": numeric_cols,
    "categorical_columns": categorical_cols,
    "missing_indicator_columns": missing_indicator_cols,
    "gender_allowed_values": sorted(list(allowed_gender_values)),
    "contract_type_allowed_values": sorted(list(allowed_contract_values)),
    "gender_invalid_values": sorted(list(gender_invalid_values)),
    "contract_type_invalid_values": sorted(list(contract_invalid_values)),
    "string_normalization_steps": [
        "strip",
        "lower",
        "empty_to_unknown",
        "na_to_unknown"
    ],
    "gender_mapping": gender_map,
    "contract_type_mapping": contract_map,
    "g4dn_used": False,
    "openshift_deploy_used": False,
    "note": "Day 2 4교시는 중복, 타입, 문자열, 범주형 값 표준화를 수행한 시간"
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
quality_output_path = OUTPUT_DIR / "day2_duplicate_type_string_quality_summary.json"
with open(quality_output_path, "w", encoding="utf-8") as f:
    json.dump(quality_summary, f, ensure_ascii=False, indent=2)
print("saved:", quality_output_path)
quality_summary


saved: /home/soldesk/ai-hybrid-lab-kim-juil/outputs/week2/day2_duplicate_type_string_quality_summary.json


{'author': 'student',
 'input_dataframe': 'df_imputed',
 'working_dataframe': 'df_quality',
 'row_count': 9,
 'full_duplicate_count': 0,
 'customer_id_duplicate_count': 0,
 'dtypes': {'customer_id': 'str',
  'age': 'int64',
  'gender': 'str',
  'monthly_fee': 'int64',
  'contract_type': 'str',
  'usage_days': 'int64',
  'support_calls': 'int64',
  'churn': 'int64',
  'age_was_missing': 'int64',
  'monthly_fee_was_missing': 'int64',
  'gender_was_missing': 'int64',
  'contract_type_was_missing': 'int64'},
 'numeric_columns': ['age', 'monthly_fee', 'usage_days', 'support_calls'],
 'categorical_columns': ['gender', 'contract_type'],
 'missing_indicator_columns': ['age_was_missing',
  'monthly_fee_was_missing',
  'gender_was_missing',
  'contract_type_was_missing'],
 'gender_allowed_values': ['female', 'male', 'unknown'],
 'contract_type_allowed_values': ['1-year',
  '2-year',
  'month-to-month',
  'unknown'],
 'gender_invalid_values': [],
 'contract_type_invalid_values': [],
 'string_norm

In [73]:
quality_candidate_path = CLEAN_DIR / "customers_quality_checked_candidate_v1.csv"
df_quality.to_csv(quality_candidate_path, index=False)
print("saved:", quality_candidate_path)
print("exists:", quality_candidate_path.exists())


saved: /home/soldesk/ai-hybrid-lab-kim-juil/data/clean/customers_quality_checked_candidate_v1.csv
exists: True


In [74]:
print("df_quality 크기:", df_quality.shape)
df_quality.head()


df_quality 크기: (9, 12)


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
0,C001,34,female,59000,month-to-month,120,1,0,0,0,0,0
1,C002,45,male,79000,1-year,380,3,1,0,0,0,0
2,C003,41,female,72000,month-to-month,45,0,0,1,1,0,0
3,C004,52,male,99000,1-year,800,5,1,0,0,0,0
4,C005,41,female,69000,unknown,220,2,0,0,0,0,1


In [75]:
df_quality.columns

Index(['customer_id', 'age', 'gender', 'monthly_fee', 'contract_type',
       'usage_days', 'support_calls', 'churn', 'age_was_missing',
       'monthly_fee_was_missing', 'gender_was_missing',
       'contract_type_was_missing'],
      dtype='str')

In [76]:
df_date = df_quality.copy()
print("df_date 크기:", df_date.shape)
df_date.head()


df_date 크기: (9, 12)


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing
0,C001,34,female,59000,month-to-month,120,1,0,0,0,0,0
1,C002,45,male,79000,1-year,380,3,1,0,0,0,0
2,C003,41,female,72000,month-to-month,45,0,0,1,1,0,0
3,C004,52,male,99000,1-year,800,5,1,0,0,0,0
4,C005,41,female,69000,unknown,220,2,0,0,0,0,1


In [77]:
print("행 수:", df_date.shape[0])

행 수: 9


In [78]:
signup_date_values = [
    "2025-01-10",
    "2024-12-01",
    "2026-04-20",
    "2023-05-15",
    "2025-08-03",
    "2025-03-18",
    "2022-11-30",
    "2026-05-10",
    "2021-09-25",
    "2024-02-14"
]

last_login_date_values = [
    "2026-06-01",
    "2026-05-10",
    "2026-06-03",
    "2026-03-01",
    "2026-05-30",
    "2026-04-15",
    "2026-01-20",
    "2026-06-04",
    "2025-12-31",
    "2026-02-28"
]


In [79]:
row_count = df_date.shape[0]
if row_count > len(signup_date_values):
    raise ValueError("실습용 날짜 리스트보다 df_date 행 수가 많습니다. 날짜 값을 추가해야 합니다.")
df_date["signup_date"] = signup_date_values[:row_count]
df_date["last_login_date"] = last_login_date_values[:row_count]
df_date[["customer_id", "signup_date", "last_login_date"]].head()

,customer_id,signup_date,last_login_date
0,C001,2025-01-10,2026-06-01
1,C002,2024-12-01,2026-05-10
2,C003,2026-04-20,2026-06-03
3,C004,2023-05-15,2026-03-01
4,C005,2025-08-03,2026-05-30


In [80]:
df_date.shape[0]

9

In [81]:
df_date.shape[1]

14

In [82]:
df_date[["signup_date", "last_login_date"]].dtypes

signup_date        str
last_login_date    str
dtype: object

In [83]:
df_date[["customer_id", "signup_date", "last_login_date"]].head()

,customer_id,signup_date,last_login_date
0,C001,2025-01-10,2026-06-01
1,C002,2024-12-01,2026-05-10
2,C003,2026-04-20,2026-06-03
3,C004,2023-05-15,2026-03-01
4,C005,2025-08-03,2026-05-30


In [84]:
df_date["signup_date"] = pd.to_datetime(df_date["signup_date"], errors="coerce")
df_date["last_login_date"] = pd.to_datetime(df_date["last_login_date"], errors="coerce")
df_date[["signup_date", "last_login_date"]].dtypes

signup_date        datetime64[us]
last_login_date    datetime64[us]
dtype: object

In [85]:
df_date["signup_date"] = pd.to_datetime(df_date["signup_date"], errors="coerce").astype("datetime64[ns]")
df_date["last_login_date"] = pd.to_datetime(df_date["last_login_date"], errors="coerce").astype("datetime64[ns]")


In [86]:
df_date[["signup_date", "last_login_date"]].dtypes

signup_date        datetime64[ns]
last_login_date    datetime64[ns]
dtype: object

In [87]:
date_error_demo = pd.Series([
    "2026-06-05",
    "2026-13-40",
    "not-a-date",
    None
])
converted_demo = pd.to_datetime(date_error_demo, errors="coerce")
pd.DataFrame({
    "raw_value": date_error_demo,
    "converted_value": converted_demo,
    "is_nat": converted_demo.isna()
})


,raw_value,converted_value,is_nat
0,2026-06-05,2026-06-05,False
1,2026-13-40,NaT,True
2,not-a-date,NaT,True
3,NaN,NaT,True


In [88]:
date_nat_summary = df_date[["signup_date", "last_login_date"]].isna().sum()
date_nat_summary

signup_date        0
last_login_date    0
dtype: int64

In [89]:
df_date[
    df_date["signup_date"].isna() | df_date["last_login_date"].isna()
][["customer_id", "signup_date", "last_login_date"]]


,customer_id,signup_date,last_login_date


In [90]:
REFERENCE_DATE = pd.Timestamp("2026-06-05")
REFERENCE_DATE

Timestamp('2026-06-05 00:00:00')

In [91]:
df_date["signup_year"] = df_date["signup_date"].dt.year
df_date[["customer_id", "signup_date", "signup_year"]].head()

,customer_id,signup_date,signup_year
0,C001,2025-01-10,2025
1,C002,2024-12-01,2024
2,C003,2026-04-20,2026
3,C004,2023-05-15,2023
4,C005,2025-08-03,2025


In [92]:
df_date["signup_month"] = df_date["signup_date"].dt.month
df_date[["customer_id", "signup_date", "signup_month"]].head()

,customer_id,signup_date,signup_month
0,C001,2025-01-10,1
1,C002,2024-12-01,12
2,C003,2026-04-20,4
3,C004,2023-05-15,5
4,C005,2025-08-03,8


In [93]:
df_date["last_login_dayofweek"] = df_date["last_login_date"].dt.dayofweek
df_date[["customer_id", "last_login_date", "last_login_dayofweek"]].head()

,customer_id,last_login_date,last_login_dayofweek
0,C001,2026-06-01,0
1,C002,2026-05-10,6
2,C003,2026-06-03,2
3,C004,2026-03-01,6
4,C005,2026-05-30,5


In [94]:
df_date["last_login_day_name"] = df_date["last_login_date"].dt.day_name()
df_date[["customer_id", "last_login_date", "last_login_day_name"]].head()

,customer_id,last_login_date,last_login_day_name
0,C001,2026-06-01,Monday
1,C002,2026-05-10,Sunday
2,C003,2026-06-03,Wednesday
3,C004,2026-03-01,Sunday
4,C005,2026-05-30,Saturday


In [95]:
df_date["days_since_signup"] = (
    REFERENCE_DATE - df_date["signup_date"]
).dt.days
df_date[["customer_id", "signup_date", "days_since_signup"]].head()

,customer_id,signup_date,days_since_signup
0,C001,2025-01-10,511
1,C002,2024-12-01,551
2,C003,2026-04-20,46
3,C004,2023-05-15,1117
4,C005,2025-08-03,306


In [96]:
df_date["days_since_last_login"] = (
    REFERENCE_DATE - df_date["last_login_date"]
).dt.days
df_date[["customer_id", "last_login_date", "days_since_last_login"]].head()

,customer_id,last_login_date,days_since_last_login
0,C001,2026-06-01,4
1,C002,2026-05-10,26
2,C003,2026-06-03,2
3,C004,2026-03-01,96
4,C005,2026-05-30,6


In [97]:
date_feature_cols = [
    "customer_id",
    "signup_date",
    "signup_year",
    "signup_month",
    "last_login_date",
    "last_login_dayofweek",
    "last_login_day_name",
    "days_since_signup",
    "days_since_last_login"
]
df_date[date_feature_cols]

,customer_id,signup_date,signup_year,signup_month,last_login_date,last_login_dayofweek,last_login_day_name,days_since_signup,days_since_last_login
0,C001,2025-01-10,2025,1,2026-06-01,0,Monday,511,4
1,C002,2024-12-01,2024,12,2026-05-10,6,Sunday,551,26
2,C003,2026-04-20,2026,4,2026-06-03,2,Wednesday,46,2
3,C004,2023-05-15,2023,5,2026-03-01,6,Sunday,1117,96
4,C005,2025-08-03,2025,8,2026-05-30,5,Saturday,306,6
6,C006,2025-03-18,2025,3,2026-04-15,2,Wednesday,444,51
7,C007,2022-11-30,2022,11,2026-01-20,1,Tuesday,1283,136
8,C008,2026-05-10,2026,5,2026-06-04,3,Thursday,26,1
9,C009,2021-09-25,2021,9,2025-12-31,2,Wednesday,1714,156


In [98]:
signup_month_counts = df_date["signup_month"].value_counts().sort_index()
signup_month_counts

signup_month
1     1
3     1
4     1
5     2
8     1
9     1
11    1
12    1
Name: count, dtype: int64

In [99]:
signup_month_counts = (
 df_date["signup_month"]
 .value_counts()
 .sort_index()
 .rename_axis("signup_month")
 .reset_index(name="customer_count")
)

signup_month_counts

,signup_month,customer_count
0,1,1
1,3,1
2,4,1
3,5,2
4,8,1
5,9,1
6,11,1
7,12,1


In [100]:
avg_days_since_last_login = df_date["days_since_last_login"].mean()
print("마지막 로그인 후 평균 경과일:", avg_days_since_last_login)

마지막 로그인 후 평균 경과일: 53.111111111111114


In [101]:
login_churn_summary = (
    df_date
    .groupby("churn")["days_since_last_login"]
    .mean()
    .reset_index()
)
login_churn_summary = login_churn_summary.rename(columns={
    "days_since_last_login": "avg_days_since_last_login"
})
login_churn_summary

,churn,avg_days_since_last_login
0,0,12.8
1,1,103.5


In [102]:
date_quality_summary = df_date[
    [
        "days_since_signup",
        "days_since_last_login"
    ]
].describe()
date_quality_summary

,days_since_signup,days_since_last_login
count,9.000000,9.000000
mean,666.444444,53.111111
std,580.142894,61.232027
min,26.000000,1.000000
25%,306.000000,4.000000
50%,511.000000,26.000000
75%,1117.000000,96.000000
max,1714.000000,156.000000


In [103]:
negative_date_features = df_date[
    (df_date["days_since_signup"] < 0) |
    (df_date["days_since_last_login"] < 0)
]
negative_date_features[
    [
        "customer_id",
        "signup_date",
        "last_login_date",
        "days_since_signup",
        "days_since_last_login"
    ]
]


,customer_id,signup_date,last_login_date,days_since_signup,days_since_last_login


In [104]:
import json
from pathlib import Path
datetime_feature_summary = {
    "author": "student",
    "input_dataframe": "df_quality",
    "working_dataframe": "df_date",
    "row_count": int(len(df_date)),
    "reference_date": str(REFERENCE_DATE.date()),
    "date_columns": [
        "signup_date",
        "last_login_date"
    ],
    "date_dtypes": {
        "signup_date": str(df_date["signup_date"].dtype),
        "last_login_date": str(df_date["last_login_date"].dtype)
    },
    "nat_count": {
        "signup_date": int(df_date["signup_date"].isna().sum()),
        "last_login_date": int(df_date["last_login_date"].isna().sum())
    },
    "created_date_features": [
        "signup_year",
        "signup_month",
        "last_login_dayofweek",
        "last_login_day_name",
        "days_since_signup",
        "days_since_last_login"
    ],
    "negative_days_since_signup_count": int((df_date["days_since_signup"] < 0).sum()),
    "negative_days_since_last_login_count": int((df_date["days_since_last_login"] < 0).sum()),
    "avg_days_since_last_login": float(df_date["days_since_last_login"].mean()),
    "g4dn_used": False,
    "openshift_deploy_used": False,
    "note": "Day 2 5교시는 문자열 날짜를 datetime으로 변환하고 날짜 파생 컬럼을 생성한 시간"
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
datetime_output_path = OUTPUT_DIR / "day2_datetime_feature_summary.json"
with open(datetime_output_path, "w", encoding="utf-8") as f:
    json.dump(datetime_feature_summary, f, ensure_ascii=False, indent=2)
print("saved:", datetime_output_path)
datetime_feature_summary


saved: /home/soldesk/ai-hybrid-lab-kim-juil/outputs/week2/day2_datetime_feature_summary.json


{'author': 'student',
 'input_dataframe': 'df_quality',
 'working_dataframe': 'df_date',
 'row_count': 9,
 'reference_date': '2026-06-05',
 'date_columns': ['signup_date', 'last_login_date'],
 'date_dtypes': {'signup_date': 'datetime64[ns]',
  'last_login_date': 'datetime64[ns]'},
 'nat_count': {'signup_date': 0, 'last_login_date': 0},
 'created_date_features': ['signup_year',
  'signup_month',
  'last_login_dayofweek',
  'last_login_day_name',
  'days_since_signup',
  'days_since_last_login'],
 'negative_days_since_signup_count': 0,
 'negative_days_since_last_login_count': 0,
 'avg_days_since_last_login': 53.111111111111114,
 'g4dn_used': False,
 'openshift_deploy_used': False,
 'note': 'Day 2 5교시는 문자열 날짜를 datetime으로 변환하고 날짜 파생 컬럼을 생성한 시간'}

In [105]:
datetime_candidate_path = CLEAN_DIR / "customers_datetime_features_candidate_v1.csv"
df_date.to_csv(datetime_candidate_path, index=False)
print("saved:", datetime_candidate_path)
print("exists:", datetime_candidate_path.exists())


saved: /home/soldesk/ai-hybrid-lab-kim-juil/data/clean/customers_datetime_features_candidate_v1.csv
exists: True


In [106]:
print("df_date 크기:", df_date.shape)
df_date.head()


df_date 크기: (9, 20)


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing,signup_date,last_login_date,signup_year,signup_month,last_login_dayofweek,last_login_day_name,days_since_signup,days_since_last_login
0,C001,34,female,59000,month-to-month,120,1,0,0,0,0,0,2025-01-10,2026-06-01,2025,1,0,Monday,511,4
1,C002,45,male,79000,1-year,380,3,1,0,0,0,0,2024-12-01,2026-05-10,2024,12,6,Sunday,551,26
2,C003,41,female,72000,month-to-month,45,0,0,1,1,0,0,2026-04-20,2026-06-03,2026,4,2,Wednesday,46,2
3,C004,52,male,99000,1-year,800,5,1,0,0,0,0,2023-05-15,2026-03-01,2023,5,6,Sunday,1117,96
4,C005,41,female,69000,unknown,220,2,0,0,0,0,1,2025-08-03,2026-05-30,2025,8,5,Saturday,306,6


In [107]:
df_date.columns

Index(['customer_id', 'age', 'gender', 'monthly_fee', 'contract_type',
       'usage_days', 'support_calls', 'churn', 'age_was_missing',
       'monthly_fee_was_missing', 'gender_was_missing',
       'contract_type_was_missing', 'signup_date', 'last_login_date',
       'signup_year', 'signup_month', 'last_login_dayofweek',
       'last_login_day_name', 'days_since_signup', 'days_since_last_login'],
      dtype='str')

In [108]:
df_outlier = df_date.copy()
print("df_outlier 크기:", df_outlier.shape)
df_outlier.head()


df_outlier 크기: (9, 20)


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing,signup_date,last_login_date,signup_year,signup_month,last_login_dayofweek,last_login_day_name,days_since_signup,days_since_last_login
0,C001,34,female,59000,month-to-month,120,1,0,0,0,0,0,2025-01-10,2026-06-01,2025,1,0,Monday,511,4
1,C002,45,male,79000,1-year,380,3,1,0,0,0,0,2024-12-01,2026-05-10,2024,12,6,Sunday,551,26
2,C003,41,female,72000,month-to-month,45,0,0,1,1,0,0,2026-04-20,2026-06-03,2026,4,2,Wednesday,46,2
3,C004,52,male,99000,1-year,800,5,1,0,0,0,0,2023-05-15,2026-03-01,2023,5,6,Sunday,1117,96
4,C005,41,female,69000,unknown,220,2,0,0,0,0,1,2025-08-03,2026-05-30,2025,8,5,Saturday,306,6


In [109]:
numeric_cols = [
    "age",
    "monthly_fee",
    "usage_days",
    "support_calls"
]
numeric_cols

['age', 'monthly_fee', 'usage_days', 'support_calls']

In [110]:
df_outlier[numeric_cols].describe()

,age,monthly_fee,usage_days,support_calls
count,9.000000,9.000000,9.000000,9.000000
mean,43.777778,80555.555556,418.888889,3.111111
std,12.397356,24429.035547,431.326281,2.571208
min,23.000000,45000.000000,15.000000,0.000000
25%,37.000000,69000.000000,90.000000,1.000000
50%,41.000000,72000.000000,220.000000,3.000000
75%,52.000000,99000.000000,800.000000,5.000000
max,63.000000,120000.000000,1100.000000,7.000000


In [111]:
for col in numeric_cols:
    print("=" * 60)
    print("컬럼:", col)
    print("min:", df_outlier[col].min())
    print("max:", df_outlier[col].max())


컬럼: age
min: 23
max: 63
컬럼: monthly_fee
min: 45000
max: 120000
컬럼: usage_days
min: 15
max: 1100
컬럼: support_calls
min: 0
max: 7


In [112]:
business_rule_outliers = pd.DataFrame({
    "age_out_of_range": ~df_outlier["age"].between(0, 120),
    "monthly_fee_out_of_range": ~df_outlier["monthly_fee"].between(0, 200000),
    "usage_days_out_of_range": ~df_outlier["usage_days"].between(0, 5000),
    "support_calls_out_of_range": ~df_outlier["support_calls"].between(0, 50),
})
business_rule_summary = business_rule_outliers.sum()
business_rule_summary

age_out_of_range              0
monthly_fee_out_of_range      0
usage_days_out_of_range       0
support_calls_out_of_range    0
dtype: int64

In [113]:
def detect_outliers_iqr(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    return lower_bound, upper_bound, outliers

In [114]:
lower_fee, upper_fee, outliers_fee = detect_outliers_iqr(df_outlier, "monthly_fee")
print("monthly_fee lower:", lower_fee)
print("monthly_fee upper:", upper_fee)
outliers_fee

monthly_fee lower: 24000.0
monthly_fee upper: 144000.0


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing,signup_date,last_login_date,signup_year,signup_month,last_login_dayofweek,last_login_day_name,days_since_signup,days_since_last_login


In [115]:
lower_calls, upper_calls, outliers_calls = detect_outliers_iqr(df_outlier, "support_calls")
print("support_calls lower:", lower_calls)
print("support_calls upper:", upper_calls)
outliers_calls

support_calls lower: -5.0
support_calls upper: 11.0


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,gender_was_missing,contract_type_was_missing,signup_date,last_login_date,signup_year,signup_month,last_login_dayofweek,last_login_day_name,days_since_signup,days_since_last_login


In [116]:
for col in numeric_cols:
    lower, upper, outliers = detect_outliers_iqr(df_outlier, col)
    print("=" * 60)
    print("컬럼:", col)
    print("lower:", lower)
    print("upper:", upper)
    print("이상 후보 수:", outliers.shape[0])


컬럼: age
lower: 14.5
upper: 74.5
이상 후보 수: 0
컬럼: monthly_fee
lower: 24000.0
upper: 144000.0
이상 후보 수: 0
컬럼: usage_days
lower: -975.0
upper: 1865.0
이상 후보 수: 0
컬럼: support_calls
lower: -5.0
upper: 11.0
이상 후보 수: 0


In [117]:
outlier_summary = []

for col in numeric_cols:
    lower, upper, outliers = detect_outliers_iqr(df_outlier, col)

    outlier_summary.append({
        "column": col,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": outliers.shape[0],
        "outlier_ratio_percent": outliers.shape[0] / df_outlier.shape[0] * 100
    })

outlier_summary_df = pd.DataFrame(outlier_summary)

outlier_summary_df

,column,lower_bound,upper_bound,outlier_count,outlier_ratio_percent
0,age,14.5,74.5,0,0.0
1,monthly_fee,24000.0,144000.0,0,0.0
2,usage_days,-975.0,1865.0,0,0.0
3,support_calls,-5.0,11.0,0,0.0


In [118]:
df_outlier_demo = df_outlier.copy()
df_outlier_demo.loc[df_outlier_demo.index[0], "monthly_fee"] = 999999
df_outlier_demo.loc[df_outlier_demo.index[1], "support_calls"] = 100
df_outlier_demo[
    [
        "customer_id",
        "monthly_fee",
        "support_calls"
    ]
].head()


,customer_id,monthly_fee,support_calls
0,C001,999999,1
1,C002,79000,100
2,C003,72000,0
3,C004,99000,5
4,C005,69000,2


In [119]:
lower_fee_demo, upper_fee_demo, outliers_fee_demo = detect_outliers_iqr(
    df_outlier_demo,
    "monthly_fee"
)
print("monthly_fee lower:", lower_fee_demo)
print("monthly_fee upper:", upper_fee_demo)
outliers_fee_demo[["customer_id", "monthly_fee"]]

monthly_fee lower: 15000.0
monthly_fee upper: 167000.0


,customer_id,monthly_fee
0,C001,999999


In [120]:
lower_calls_demo, upper_calls_demo, outliers_calls_demo = detect_outliers_iqr(
    df_outlier_demo,
    "support_calls"
)
print("support_calls lower:", lower_calls_demo)
print("support_calls upper:", upper_calls_demo)
outliers_calls_demo[["customer_id", "support_calls"]]

support_calls lower: -6.5
support_calls upper: 13.5


,customer_id,support_calls
1,C002,100


In [121]:
lower_fee, upper_fee, outliers_fee = detect_outliers_iqr(df_outlier, "monthly_fee")
df_outlier["monthly_fee_outlier"] = (
    (df_outlier["monthly_fee"] < lower_fee) |
    (df_outlier["monthly_fee"] > upper_fee)
).astype(int)
df_outlier[
    [
        "customer_id",
        "monthly_fee",
        "monthly_fee_outlier"
    ]
].head()


,customer_id,monthly_fee,monthly_fee_outlier
0,C001,59000,0
1,C002,79000,0
2,C003,72000,0
3,C004,99000,0
4,C005,69000,0


In [122]:
lower_calls, upper_calls, outliers_calls = detect_outliers_iqr(df_outlier, "support_calls")
df_outlier["support_calls_outlier"] = (
    (df_outlier["support_calls"] < lower_calls) |
    (df_outlier["support_calls"] > upper_calls)
).astype(int)
df_outlier[
    [
        "customer_id",
        "support_calls",
        "support_calls_outlier"
    ]
].head()


,customer_id,support_calls,support_calls_outlier
0,C001,1,0
1,C002,3,0
2,C003,0,0
3,C004,5,0
4,C005,2,0


In [123]:
df_clipping_demo = df_outlier.copy()
df_clipping_demo["monthly_fee_clipped"] = df_clipping_demo["monthly_fee"].clip(
    lower=lower_fee,
    upper=upper_fee
)
df_clipping_demo[
    [
        "customer_id",
        "monthly_fee",
        "monthly_fee_clipped"
    ]
].head()


,customer_id,monthly_fee,monthly_fee_clipped
0,C001,59000,59000
1,C002,79000,79000
2,C003,72000,72000
3,C004,99000,99000
4,C005,69000,69000


In [124]:
flag_summary = []
for col in ["monthly_fee_outlier", "support_calls_outlier"]:
    counts = df_outlier[col].value_counts(dropna=False).to_dict()
    flag_summary.append({
        "flag_column": col,
        "normal_count": int(counts.get(0, 0)),
        "outlier_count": int(counts.get(1, 0))
    })
flag_summary_df = pd.DataFrame(flag_summary)
flag_summary_df

,flag_column,normal_count,outlier_count
0,monthly_fee_outlier,9,0
1,support_calls_outlier,9,0


In [125]:
import json
from pathlib import Path
numeric_outlier_summary = {
    "author": "student",
    "input_dataframe": "df_date",
    "working_dataframe": "df_outlier",
    "row_count": int(len(df_outlier)),
    "numeric_columns": numeric_cols,
    "business_rule_summary": {
        key: int(value)
        for key, value in business_rule_summary.items()
    },
    "iqr_outlier_summary": outlier_summary_df.to_dict(orient="records"),
    "outlier_flag_columns": [
        "monthly_fee_outlier",
        "support_calls_outlier"
    ],
    "outlier_flag_summary": flag_summary_df.to_dict(orient="records"),
    "selected_strategy": "flag_only",
    "reason": "데이터 수가 적고 이상 후보가 실제 특이 고객일 수 있으므로 바로 삭제하지 않고 플래그로 기록한다.",
    "g4dn_used": False,
    "openshift_deploy_used": False,
    "note": "Day 2 6교시는 수치형 컬럼의 이상 후보를 IQR로 점검하고 이상 플래그를 생성한 시간"
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
numeric_outlier_output_path = OUTPUT_DIR / "day2_numeric_outlier_summary.json"
with open(numeric_outlier_output_path, "w", encoding="utf-8") as f:
    json.dump(numeric_outlier_summary, f, ensure_ascii=False, indent=2)
print("saved:", numeric_outlier_output_path)
numeric_outlier_summary


saved: /home/soldesk/ai-hybrid-lab-kim-juil/outputs/week2/day2_numeric_outlier_summary.json


{'author': 'student',
 'input_dataframe': 'df_date',
 'working_dataframe': 'df_outlier',
 'row_count': 9,
 'numeric_columns': ['age', 'monthly_fee', 'usage_days', 'support_calls'],
 'business_rule_summary': {'age_out_of_range': 0,
  'monthly_fee_out_of_range': 0,
  'usage_days_out_of_range': 0,
  'support_calls_out_of_range': 0},
 'iqr_outlier_summary': [{'column': 'age',
   'lower_bound': 14.5,
   'upper_bound': 74.5,
   'outlier_count': 0,
   'outlier_ratio_percent': 0.0},
  {'column': 'monthly_fee',
   'lower_bound': 24000.0,
   'upper_bound': 144000.0,
   'outlier_count': 0,
   'outlier_ratio_percent': 0.0},
  {'column': 'usage_days',
   'lower_bound': -975.0,
   'upper_bound': 1865.0,
   'outlier_count': 0,
   'outlier_ratio_percent': 0.0},
  {'column': 'support_calls',
   'lower_bound': -5.0,
   'upper_bound': 11.0,
   'outlier_count': 0,
   'outlier_ratio_percent': 0.0}],
 'outlier_flag_columns': ['monthly_fee_outlier', 'support_calls_outlier'],
 'outlier_flag_summary': [{'flag_

In [126]:
outlier_candidate_path = CLEAN_DIR / "customers_outlier_checked_candidate_v1.csv"
df_outlier.to_csv(outlier_candidate_path, index=False)
print("saved:", outlier_candidate_path)
print("exists:", outlier_candidate_path.exists())


saved: /home/soldesk/ai-hybrid-lab-kim-juil/data/clean/customers_outlier_checked_candidate_v1.csv
exists: True


In [127]:
print("df_outlier 크기:", df_outlier.shape)
df_outlier.head()

df_outlier 크기: (9, 22)


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,...,signup_date,last_login_date,signup_year,signup_month,last_login_dayofweek,last_login_day_name,days_since_signup,days_since_last_login,monthly_fee_outlier,support_calls_outlier
0,C001,34,female,59000,month-to-month,120,1,0,0,0,...,2025-01-10,2026-06-01,2025,1,0,Monday,511,4,0,0
1,C002,45,male,79000,1-year,380,3,1,0,0,...,2024-12-01,2026-05-10,2024,12,6,Sunday,551,26,0,0
2,C003,41,female,72000,month-to-month,45,0,0,1,1,...,2026-04-20,2026-06-03,2026,4,2,Wednesday,46,2,0,0
3,C004,52,male,99000,1-year,800,5,1,0,0,...,2023-05-15,2026-03-01,2023,5,6,Sunday,1117,96,0,0
4,C005,41,female,69000,unknown,220,2,0,0,0,...,2025-08-03,2026-05-30,2025,8,5,Saturday,306,6,0,0


In [128]:
df_outlier.columns

Index(['customer_id', 'age', 'gender', 'monthly_fee', 'contract_type',
       'usage_days', 'support_calls', 'churn', 'age_was_missing',
       'monthly_fee_was_missing', 'gender_was_missing',
       'contract_type_was_missing', 'signup_date', 'last_login_date',
       'signup_year', 'signup_month', 'last_login_dayofweek',
       'last_login_day_name', 'days_since_signup', 'days_since_last_login',
       'monthly_fee_outlier', 'support_calls_outlier'],
      dtype='str')

In [129]:
print("df 존재:", "df" in globals())
print("df_date 존재:", "df_date" in globals())
print("df_outlier 존재:", "df_outlier" in globals())

df 존재: False
df_date 존재: True
df_outlier 존재: True


In [130]:
df_feature = df_outlier.copy()
print("df_feature 크기:", df_feature.shape)
df_feature.head()


df_feature 크기: (9, 22)


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,...,signup_date,last_login_date,signup_year,signup_month,last_login_dayofweek,last_login_day_name,days_since_signup,days_since_last_login,monthly_fee_outlier,support_calls_outlier
0,C001,34,female,59000,month-to-month,120,1,0,0,0,...,2025-01-10,2026-06-01,2025,1,0,Monday,511,4,0,0
1,C002,45,male,79000,1-year,380,3,1,0,0,...,2024-12-01,2026-05-10,2024,12,6,Sunday,551,26,0,0
2,C003,41,female,72000,month-to-month,45,0,0,1,1,...,2026-04-20,2026-06-03,2026,4,2,Wednesday,46,2,0,0
3,C004,52,male,99000,1-year,800,5,1,0,0,...,2023-05-15,2026-03-01,2023,5,6,Sunday,1117,96,0,0
4,C005,41,female,69000,unknown,220,2,0,0,0,...,2025-08-03,2026-05-30,2025,8,5,Saturday,306,6,0,0


In [131]:
def standardize_gender(value):
    if pd.isna(value):
        return "unknown"
    
    value = str(value).strip().lower()
    
    gender_map = {
        "m": "male",
        "male": "male",
        "f": "female",
        "female": "female",
        "unknown": "unknown"
    }
    
    return gender_map.get(value, "unknown")

df_feature["gender_standard"] = df_feature["gender"].apply(standardize_gender)

df_feature[["customer_id", "gender", "gender_standard"]].head()

,customer_id,gender,gender_standard
0,C001,female,female
1,C002,male,male
2,C003,female,female
3,C004,male,male
4,C005,female,female


In [132]:
def standardize_contract_type(value):
    if pd.isna(value):
        return "unknown"
    
    value = str(value).strip().lower()
    
    contract_map = {
        "monthly": "monthly",
        "month-to-month": "monthly",
        "m2m": "monthly",
        "yearly": "one_year",
        "1-year": "one_year",
        "one-year": "one_year",
        "one_year": "one_year",
        "2-year": "two_year",
        "two-year": "two_year",
        "two_year": "two_year",
        "unknown": "unknown"
    }
    
    return contract_map.get(value, "unknown")

df_feature["contract_type_standard"] = df_feature["contract_type"].apply(standardize_contract_type)
df_feature[["customer_id", "contract_type", "contract_type_standard"]].head()

,customer_id,contract_type,contract_type_standard
0,C001,month-to-month,monthly
1,C002,1-year,one_year
2,C003,month-to-month,monthly
3,C004,1-year,one_year
4,C005,unknown,unknown


In [133]:
df_feature["gender_standard"].value_counts(dropna=False)
df_feature["contract_type_standard"].value_counts(dropna=False)

contract_type_standard
monthly     4
one_year    4
unknown     1
Name: count, dtype: int64

In [134]:
gender_count_df = (
 df_feature["gender_standard"]
 .value_counts(dropna=False)
 .rename_axis("gender_standard")
 .reset_index(name="count")
)
gender_count_df

,gender_standard,count
0,female,4
1,male,4
2,unknown,1


In [135]:
contract_type_count_df = (
    df_feature["contract_type_standard"]
    .value_counts(dropna=False)
    .rename_axis("contract_type_standard")
    .reset_index(name="count")
)
contract_type_count_df

,contract_type_standard,count
0,monthly,4
1,one_year,4
2,unknown,1


In [136]:
allowed_gender = ["male", "female", "unknown"]
allowed_contract_type = [
    "monthly",
    "one_year",
    "two_year",
    "unknown"
]
print("allowed_gender:", allowed_gender)
print("allowed_contract_type:", allowed_contract_type)


allowed_gender: ['male', 'female', 'unknown']
allowed_contract_type: ['monthly', 'one_year', 'two_year', 'unknown']


In [137]:
actual_gender_values = set(df_feature["gender_standard"].dropna().unique())
actual_contract_values = set(df_feature["contract_type_standard"].dropna().unique())
invalid_gender_values = actual_gender_values - set(allowed_gender)
invalid_contract_values = actual_contract_values - set(allowed_contract_type)
print("gender 허용 외 값:", invalid_gender_values)
print("contract_type 허용 외 값:", invalid_contract_values)


gender 허용 외 값: set()
contract_type 허용 외 값: set()


In [138]:
df_category_demo = df_feature.copy()
df_category_demo.loc[df_category_demo.index[0], "gender_standard"] = "woman"
df_category_demo.loc[df_category_demo.index[1], "contract_type_standard"] = "vip_plan"
demo_gender_values = set(df_category_demo["gender_standard"].dropna().unique())
demo_contract_values = set(df_category_demo["contract_type_standard"].dropna().unique())
demo_invalid_gender = demo_gender_values - set(allowed_gender)
demo_invalid_contract = demo_contract_values - set(allowed_contract_type)
print("데모 gender 허용 외 값:", demo_invalid_gender)
print("데모 contract_type 허용 외 값:", demo_invalid_contract)


데모 gender 허용 외 값: {'woman'}
데모 contract_type 허용 외 값: {'vip_plan'}


In [ ]:
df_feature["is_high_fee"] = (df_feature["monthly_fee"] >= 50000).astype(int)
df_feature[
    [
        "customer_id",
        "monthly_fee",
        "is_high_fee"
    ]
].head()


,customer_id,monthly_fee,is_high_fee
0,C001,59000,1
1,C002,79000,1
2,C003,72000,1
3,C004,99000,1
4,C005,69000,1


In [141]:
df_feature["is_high_fee"] = (df_feature["monthly_fee"] >= 50000).astype(int)
df_feature[
    [
        "customer_id",
        "monthly_fee",
        "is_high_fee"
    ]
]

,customer_id,monthly_fee,is_high_fee
0,C001,59000,1
1,C002,79000,1
2,C003,72000,1
3,C004,99000,1
4,C005,69000,1
6,C006,72000,1
7,C007,110000,1
8,C008,45000,0
9,C009,120000,1


In [142]:
df_feature["is_short_user"] = (df_feature["usage_days"] < 30).astype(int)
df_feature[
    [
        "customer_id",
        "usage_days",
        "is_short_user"
    ]
]


,customer_id,usage_days,is_short_user
0,C001,120,0
1,C002,380,0
2,C003,45,0
3,C004,800,0
4,C005,220,0
6,C006,90,0
7,C007,1000,0
8,C008,15,1
9,C009,1100,0


In [143]:
df_feature["has_many_support_calls"] = (df_feature["support_calls"] >= 3).astype(int)
df_feature[
    [
        "customer_id",
        "support_calls",
        "has_many_support_calls"
    ]
]


,customer_id,support_calls,has_many_support_calls
0,C001,1,0
1,C002,3,1
2,C003,0,0
3,C004,5,1
4,C005,2,0
6,C006,4,1
7,C007,6,1
8,C008,0,0
9,C009,7,1


In [144]:
def usage_group(days):
    if days < 30:
        return "short"
    elif days < 365:
        return "middle"
    else:
        return "long"


In [145]:
print(usage_group(10))
print(usage_group(120))
print(usage_group(800))


short
middle
long


In [146]:
df_feature["usage_group"] = df_feature["usage_days"].apply(usage_group)
df_feature[
    [
        "customer_id",
        "usage_days",
        "usage_group"
    ]
]


,customer_id,usage_days,usage_group
0,C001,120,middle
1,C002,380,long
2,C003,45,middle
3,C004,800,long
4,C005,220,middle
6,C006,90,middle
7,C007,1000,long
8,C008,15,short
9,C009,1100,long


In [147]:
df_feature["usage_group"].value_counts(dropna=False)

usage_group
middle    4
long      4
short     1
Name: count, dtype: int64

In [148]:
df_feature["is_long_contract"] = (
    df_feature["contract_type_standard"].isin(["one_year", "two_year"])
).astype(int)
df_feature[
    [
        "customer_id",
        "contract_type_standard",
        "is_long_contract"
    ]
]


,customer_id,contract_type_standard,is_long_contract
0,C001,monthly,0
1,C002,one_year,1
2,C003,monthly,0
3,C004,one_year,1
4,C005,unknown,0
6,C006,monthly,0
7,C007,one_year,1
8,C008,monthly,0
9,C009,one_year,1


In [149]:
feature_cols_created = [
    "is_high_fee",
    "is_short_user",
    "has_many_support_calls",
    "usage_group",
    "is_long_contract"
]
df_feature[
    [
        "customer_id",
        "monthly_fee",
        "usage_days",
        "support_calls",
        "contract_type_standard"
    ] + feature_cols_created
]


,customer_id,monthly_fee,usage_days,support_calls,contract_type_standard,is_high_fee,is_short_user,has_many_support_calls,usage_group,is_long_contract
0,C001,59000,120,1,monthly,1,0,0,middle,0
1,C002,79000,380,3,one_year,1,0,1,long,1
2,C003,72000,45,0,monthly,1,0,0,middle,0
3,C004,99000,800,5,one_year,1,0,1,long,1
4,C005,69000,220,2,unknown,1,0,0,middle,0
6,C006,72000,90,4,monthly,1,0,1,middle,0
7,C007,110000,1000,6,one_year,1,0,1,long,1
8,C008,45000,15,0,monthly,0,1,0,short,0
9,C009,120000,1100,7,one_year,1,0,1,long,1


In [150]:
for col in feature_cols_created:
    print("=" * 60)
    print("컬럼:", col)
    print(df_feature[col].value_counts(dropna=False))


컬럼: is_high_fee
is_high_fee
1    8
0    1
Name: count, dtype: int64
컬럼: is_short_user
is_short_user
0    8
1    1
Name: count, dtype: int64
컬럼: has_many_support_calls
has_many_support_calls
1    5
0    4
Name: count, dtype: int64
컬럼: usage_group
usage_group
middle    4
long      4
short     1
Name: count, dtype: int64
컬럼: is_long_contract
is_long_contract
0    5
1    4
Name: count, dtype: int64


In [151]:
for col in feature_cols_created:
    print("=" * 60)
    print("컬럼:", col)
    
    count_df = (
        df_feature[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="count")
    )
    
    display(count_df)

컬럼: is_high_fee


,is_high_fee,count
0,1,8
1,0,1


컬럼: is_short_user


,is_short_user,count
0,0,8
1,1,1


컬럼: has_many_support_calls


,has_many_support_calls,count
0,1,5
1,0,4


컬럼: usage_group


,usage_group,count
0,middle,4
1,long,4
2,short,1


컬럼: is_long_contract


,is_long_contract,count
0,0,5
1,1,4


In [152]:
for col in feature_cols_created:
    print("=" * 60)
    print("컬럼:", col)
    
    count_df = (
        df_feature[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="count")
    )
    
    print(count_df.to_string(index=False))

컬럼: is_high_fee
 is_high_fee  count
           1      8
           0      1
컬럼: is_short_user
 is_short_user  count
             0      8
             1      1
컬럼: has_many_support_calls
 has_many_support_calls  count
                      1      5
                      0      4
컬럼: usage_group
usage_group  count
     middle      4
       long      4
      short      1
컬럼: is_long_contract
 is_long_contract  count
                0      5
                1      4


In [153]:
high_fee_churn = (
    df_feature
    .groupby("is_high_fee")["churn"]
    .mean()
    .reset_index()
    .rename(columns={"churn": "churn_rate"})
)
high_fee_churn

,is_high_fee,churn_rate
0,0,0.0
1,1,0.5


In [154]:
usage_group_churn = (
    df_feature
    .groupby("usage_group")["churn"]
    .mean()
    .reset_index()
    .rename(columns={"churn": "churn_rate"})
)
usage_group_churn["churn_rate_percent"] = usage_group_churn["churn_rate"] * 100
usage_group_churn

,usage_group,churn_rate,churn_rate_percent
0,long,1.0,100.0
1,middle,0.0,0.0
2,short,0.0,0.0


In [155]:
support_churn = (
    df_feature
    .groupby("has_many_support_calls")["churn"]
    .mean()
    .reset_index()
    .rename(columns={"churn": "churn_rate"})
)
support_churn

,has_many_support_calls,churn_rate
0,0,0.0
1,1,0.8


In [156]:
df_feature["leakage_feature"] = df_feature["churn"]
df_feature[
    [
        "customer_id",
        "churn",
        "leakage_feature"
    ]
]


,customer_id,churn,leakage_feature
0,C001,0,0
1,C002,1,1
2,C003,0,0
3,C004,1,1
4,C005,0,0
6,C006,0,0
7,C007,1,1
8,C008,0,0
9,C009,1,1


In [157]:
df_feature = df_feature.drop(columns=["leakage_feature"])
print("leakage_feature" in df_feature.columns)

False


In [158]:
import json
from pathlib import Path
feature_value_summary = {}
for col in feature_cols_created:
    feature_value_summary[col] = {
        str(k): int(v)
        for k, v in df_feature[col].value_counts(dropna=False).to_dict().items()
    }
categorical_feature_summary = {
    "author": "student",
    "input_dataframe": "df_outlier",
    "working_dataframe": "df_feature",
    "row_count": int(len(df_feature)),
    "categorical_columns": [
        "gender_standard",
        "contract_type_standard"
    ],
    "allowed_gender": allowed_gender,
    "allowed_contract_type": allowed_contract_type,
    "invalid_gender_values": sorted(list(invalid_gender_values)),
    "invalid_contract_values": sorted(list(invalid_contract_values)),
    "created_features": feature_cols_created,
    "feature_creation_rules": {
        "is_high_fee": "monthly_fee >= 50000",
        "is_short_user": "usage_days < 30",
        "has_many_support_calls": "support_calls >= 3",
        "usage_group": "usage_days < 30 short, usage_days < 365 middle, else long",
        "is_long_contract": "contract_type_standard in ['one_year', 'two_year']"
    },
    "feature_value_summary": feature_value_summary,
    "leakage_demo_created": True,
    "leakage_demo_removed": "leakage_feature" not in df_feature.columns,
    "leakage_warning": "Do not create features from target column churn or future information.",
    "selected_strategy": "create_interpretable_features_without_target_or_future_information",
    "g4dn_used": False,
    "openshift_deploy_used": False,
    "note": "Day 2 7교시는 범주형 값을 최종 검증하고 파생 변수를 생성하며 데이터 누수를 설명한 시간"
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
categorical_feature_output_path = OUTPUT_DIR / "day2_categorical_feature_engineering_summary.json"
with open(categorical_feature_output_path, "w", encoding="utf-8") as f:
    json.dump(categorical_feature_summary, f, ensure_ascii=False, indent=2)
print("saved:", categorical_feature_output_path)
categorical_feature_summary


saved: /home/soldesk/ai-hybrid-lab-kim-juil/outputs/week2/day2_categorical_feature_engineering_summary.json


{'author': 'student',
 'input_dataframe': 'df_outlier',
 'working_dataframe': 'df_feature',
 'row_count': 9,
 'categorical_columns': ['gender_standard', 'contract_type_standard'],
 'allowed_gender': ['male', 'female', 'unknown'],
 'allowed_contract_type': ['monthly', 'one_year', 'two_year', 'unknown'],
 'invalid_gender_values': [],
 'invalid_contract_values': [],
 'created_features': ['is_high_fee',
  'is_short_user',
  'has_many_support_calls',
  'usage_group',
  'is_long_contract'],
 'feature_creation_rules': {'is_high_fee': 'monthly_fee >= 50000',
  'is_short_user': 'usage_days < 30',
  'has_many_support_calls': 'support_calls >= 3',
  'usage_group': 'usage_days < 30 short, usage_days < 365 middle, else long',
  'is_long_contract': "contract_type_standard in ['one_year', 'two_year']"},
 'feature_value_summary': {'is_high_fee': {'1': 8, '0': 1},
  'is_short_user': {'0': 8, '1': 1},
  'has_many_support_calls': {'1': 5, '0': 4},
  'usage_group': {'middle': 4, 'long': 4, 'short': 1},
  

In [159]:
feature_candidate_path = CLEAN_DIR / "customers_feature_engineering_candidate_v1.csv"
df_feature.to_csv(feature_candidate_path, index=False)
print("saved:", feature_candidate_path)
print("exists:", feature_candidate_path.exists())


saved: /home/soldesk/ai-hybrid-lab-kim-juil/data/clean/customers_feature_engineering_candidate_v1.csv
exists: True


In [160]:
print("df_feature 크기:", df_feature.shape)
df_feature.head()


df_feature 크기: (9, 29)


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,...,days_since_last_login,monthly_fee_outlier,support_calls_outlier,gender_standard,contract_type_standard,is_high_fee,is_short_user,has_many_support_calls,usage_group,is_long_contract
0,C001,34,female,59000,month-to-month,120,1,0,0,0,...,4,0,0,female,monthly,1,0,0,middle,0
1,C002,45,male,79000,1-year,380,3,1,0,0,...,26,0,0,male,one_year,1,0,1,long,1
2,C003,41,female,72000,month-to-month,45,0,0,1,1,...,2,0,0,female,monthly,1,0,0,middle,0
3,C004,52,male,99000,1-year,800,5,1,0,0,...,96,0,0,male,one_year,1,0,1,long,1
4,C005,41,female,69000,unknown,220,2,0,0,0,...,6,0,0,female,unknown,1,0,0,middle,0


In [161]:
df_feature.columns

Index(['customer_id', 'age', 'gender', 'monthly_fee', 'contract_type',
       'usage_days', 'support_calls', 'churn', 'age_was_missing',
       'monthly_fee_was_missing', 'gender_was_missing',
       'contract_type_was_missing', 'signup_date', 'last_login_date',
       'signup_year', 'signup_month', 'last_login_dayofweek',
       'last_login_day_name', 'days_since_signup', 'days_since_last_login',
       'monthly_fee_outlier', 'support_calls_outlier', 'gender_standard',
       'contract_type_standard', 'is_high_fee', 'is_short_user',
       'has_many_support_calls', 'usage_group', 'is_long_contract'],
      dtype='str')

In [162]:
df_clean_v2 = df_feature.copy()
print("df_clean_v2 크기:", df_clean_v2.shape)
df_clean_v2.head()


df_clean_v2 크기: (9, 29)


,customer_id,age,gender,monthly_fee,contract_type,usage_days,support_calls,churn,age_was_missing,monthly_fee_was_missing,...,days_since_last_login,monthly_fee_outlier,support_calls_outlier,gender_standard,contract_type_standard,is_high_fee,is_short_user,has_many_support_calls,usage_group,is_long_contract
0,C001,34,female,59000,month-to-month,120,1,0,0,0,...,4,0,0,female,monthly,1,0,0,middle,0
1,C002,45,male,79000,1-year,380,3,1,0,0,...,26,0,0,male,one_year,1,0,1,long,1
2,C003,41,female,72000,month-to-month,45,0,0,1,1,...,2,0,0,female,monthly,1,0,0,middle,0
3,C004,52,male,99000,1-year,800,5,1,0,0,...,96,0,0,male,one_year,1,0,1,long,1
4,C005,41,female,69000,unknown,220,2,0,0,0,...,6,0,0,female,unknown,1,0,0,middle,0


In [163]:
"leakage_feature" in df_clean_v2.columns

False

In [164]:
df_clean_v2.columns.tolist()

['customer_id',
 'age',
 'gender',
 'monthly_fee',
 'contract_type',
 'usage_days',
 'support_calls',
 'churn',
 'age_was_missing',
 'monthly_fee_was_missing',
 'gender_was_missing',
 'contract_type_was_missing',
 'signup_date',
 'last_login_date',
 'signup_year',
 'signup_month',
 'last_login_dayofweek',
 'last_login_day_name',
 'days_since_signup',
 'days_since_last_login',
 'monthly_fee_outlier',
 'support_calls_outlier',
 'gender_standard',
 'contract_type_standard',
 'is_high_fee',
 'is_short_user',
 'has_many_support_calls',
 'usage_group',
 'is_long_contract']

In [165]:
preferred_columns = [
    # ID
    "customer_id",

    # 원본 수치형
    "age",
    "monthly_fee",
    "usage_days",
    "support_calls",

    # 범주형 표준값
    "gender_standard",
    "contract_type_standard",

    # 날짜 원본 및 파생
    "signup_date",
    "last_login_date",
    "signup_year",
    "signup_month",
    "last_login_dayofweek",
    "last_login_day_name",
    "days_since_signup",
    "days_since_last_login",

    # 결측 여부 플래그
    "age_was_missing",
    "monthly_fee_was_missing",
    "gender_was_missing",
    "contract_type_was_missing",

    # 이상치 플래그
    "monthly_fee_outlier",
    "support_calls_outlier",

    # 파생 변수
    "is_high_fee",
    "is_short_user",
    "has_many_support_calls",
    "usage_group",
    "is_long_contract",

    # Target
    "churn"
]

# preferred_columns 중 실제 df_clean_v2에 존재하는 컬럼만 선택
existing_preferred_columns = [
    col for col in preferred_columns
    if col in df_clean_v2.columns
]

# preferred_columns에는 없지만 df_clean_v2에 남아 있는 나머지 컬럼
remaining_columns = [
    col for col in df_clean_v2.columns
    if col not in existing_preferred_columns
]

# df_clean_v2에 없는 preferred_columns 확인
missing_preferred_columns = [
    col for col in preferred_columns
    if col not in df_clean_v2.columns
]

# 컬럼 순서 재배치
df_clean_v2 = df_clean_v2[
    existing_preferred_columns + remaining_columns
]

print("우선 배치된 컬럼 수:", len(existing_preferred_columns))
print("뒤에 유지된 나머지 컬럼 수:", len(remaining_columns))
print("df_clean_v2에 없는 preferred 컬럼:", missing_preferred_columns)

df_clean_v2.head()


우선 배치된 컬럼 수: 27
뒤에 유지된 나머지 컬럼 수: 2
df_clean_v2에 없는 preferred 컬럼: []


,customer_id,age,monthly_fee,usage_days,support_calls,gender_standard,contract_type_standard,signup_date,last_login_date,signup_year,...,monthly_fee_outlier,support_calls_outlier,is_high_fee,is_short_user,has_many_support_calls,usage_group,is_long_contract,churn,gender,contract_type
0,C001,34,59000,120,1,female,monthly,2025-01-10,2026-06-01,2025,...,0,0,1,0,0,middle,0,0,female,month-to-month
1,C002,45,79000,380,3,male,one_year,2024-12-01,2026-05-10,2024,...,0,0,1,0,1,long,1,1,male,1-year
2,C003,41,72000,45,0,female,monthly,2026-04-20,2026-06-03,2026,...,0,0,1,0,0,middle,0,0,female,month-to-month
3,C004,52,99000,800,5,male,one_year,2023-05-15,2026-03-01,2023,...,0,0,1,0,1,long,1,1,male,1-year
4,C005,41,69000,220,2,female,unknown,2025-08-03,2026-05-30,2025,...,0,0,1,0,0,middle,0,0,female,unknown


In [166]:
final_quality_summary = {
    "row_count": int(df_clean_v2.shape[0]),
    "column_count": int(df_clean_v2.shape[1]),
    "missing_total": int(df_clean_v2.isnull().sum().sum()),
    "full_duplicate_count": int(df_clean_v2.duplicated().sum()),
    "customer_id_duplicate_count": int(df_clean_v2.duplicated(subset=["customer_id"]).sum())
}
final_quality_summary

{'row_count': 9,
 'column_count': 29,
 'missing_total': 0,
 'full_duplicate_count': 0,
 'customer_id_duplicate_count': 0}

In [167]:
allowed_gender = {"male", "female", "unknown"}
allowed_contract_type = {"monthly", "one_year", "two_year", "unknown"}
allowed_usage_group = {"short", "middle", "long"}
gender_invalid_final = set(df_clean_v2["gender_standard"].dropna().unique()) - allowed_gender
contract_invalid_final = set(df_clean_v2["contract_type_standard"].dropna().unique()) - allowed_contract_type
usage_group_invalid_final = set(df_clean_v2["usage_group"].dropna().unique()) - allowed_usage_group
print("gender 허용 외 값:", gender_invalid_final)
print("contract_type 허용 외 값:", contract_invalid_final)
print("usage_group 허용 외 값:", usage_group_invalid_final)


gender 허용 외 값: set()
contract_type 허용 외 값: set()
usage_group 허용 외 값: set()


In [168]:
date_final_summary = {
    "signup_date_dtype": str(df_clean_v2["signup_date"].dtype),
    "last_login_date_dtype": str(df_clean_v2["last_login_date"].dtype),
    "signup_date_nat_count": int(df_clean_v2["signup_date"].isna().sum()),
    "last_login_date_nat_count": int(df_clean_v2["last_login_date"].isna().sum()),
    "negative_days_since_signup_count": int((df_clean_v2["days_since_signup"] < 0).sum()),
    "negative_days_since_last_login_count": int((df_clean_v2["days_since_last_login"] < 0).sum())
}
date_final_summary

{'signup_date_dtype': 'datetime64[ns]',
 'last_login_date_dtype': 'datetime64[ns]',
 'signup_date_nat_count': 0,
 'last_login_date_nat_count': 0,
 'negative_days_since_signup_count': 0,
 'negative_days_since_last_login_count': 0}

In [169]:
from pathlib import Path
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
clean_v2_csv = CLEAN_DIR / "customers_clean_20260606_v2.csv"
clean_v2_parquet = CLEAN_DIR / "customers_clean_20260606_v2.parquet"
print("CSV 저장 경로:", clean_v2_csv)
print("Parquet 저장 경로:", clean_v2_parquet)


CSV 저장 경로: /home/soldesk/ai-hybrid-lab-kim-juil/data/clean/customers_clean_20260606_v2.csv
Parquet 저장 경로: /home/soldesk/ai-hybrid-lab-kim-juil/data/clean/customers_clean_20260606_v2.parquet


In [170]:
df_clean_v2.to_csv(clean_v2_csv, index=False)
print("CSV 저장 완료:", clean_v2_csv)
print("CSV 존재 여부:", clean_v2_csv.exists())


CSV 저장 완료: /home/soldesk/ai-hybrid-lab-kim-juil/data/clean/customers_clean_20260606_v2.csv
CSV 존재 여부: True


In [171]:
import pyarrow
print("pyarrow:", pyarrow.__version__)


pyarrow: 24.0.0


In [172]:
df_clean_v2.to_parquet(clean_v2_parquet, index=False)
print("Parquet 저장 완료:", clean_v2_parquet)
print("Parquet 존재 여부:", clean_v2_parquet.exists())


Parquet 저장 완료: /home/soldesk/ai-hybrid-lab-kim-juil/data/clean/customers_clean_20260606_v2.parquet
Parquet 존재 여부: True


In [173]:
df_csv_check = pd.read_csv(clean_v2_csv)
print("CSV 다시 읽기 shape:", df_csv_check.shape)
df_csv_check.head()


CSV 다시 읽기 shape: (9, 29)


,customer_id,age,monthly_fee,usage_days,support_calls,gender_standard,contract_type_standard,signup_date,last_login_date,signup_year,...,monthly_fee_outlier,support_calls_outlier,is_high_fee,is_short_user,has_many_support_calls,usage_group,is_long_contract,churn,gender,contract_type
0,C001,34,59000,120,1,female,monthly,2025-01-10,2026-06-01,2025,...,0,0,1,0,0,middle,0,0,female,month-to-month
1,C002,45,79000,380,3,male,one_year,2024-12-01,2026-05-10,2024,...,0,0,1,0,1,long,1,1,male,1-year
2,C003,41,72000,45,0,female,monthly,2026-04-20,2026-06-03,2026,...,0,0,1,0,0,middle,0,0,female,month-to-month
3,C004,52,99000,800,5,male,one_year,2023-05-15,2026-03-01,2023,...,0,0,1,0,1,long,1,1,male,1-year
4,C005,41,69000,220,2,female,unknown,2025-08-03,2026-05-30,2025,...,0,0,1,0,0,middle,0,0,female,unknown


In [174]:
df_csv_check.dtypes

customer_id                    str
age                          int64
monthly_fee                  int64
usage_days                   int64
support_calls                int64
gender_standard                str
contract_type_standard         str
signup_date                    str
last_login_date                str
signup_year                  int64
signup_month                 int64
last_login_dayofweek         int64
last_login_day_name            str
days_since_signup            int64
days_since_last_login        int64
age_was_missing              int64
monthly_fee_was_missing      int64
gender_was_missing           int64
contract_type_was_missing    int64
monthly_fee_outlier          int64
support_calls_outlier        int64
is_high_fee                  int64
is_short_user                int64
has_many_support_calls       int64
usage_group                    str
is_long_contract             int64
churn                        int64
gender                         str
contract_type       

In [175]:
df_parquet_check = pd.read_parquet(clean_v2_parquet)
print("Parquet 다시 읽기 shape:", df_parquet_check.shape)
df_parquet_check.head()


Parquet 다시 읽기 shape: (9, 29)


,customer_id,age,monthly_fee,usage_days,support_calls,gender_standard,contract_type_standard,signup_date,last_login_date,signup_year,...,monthly_fee_outlier,support_calls_outlier,is_high_fee,is_short_user,has_many_support_calls,usage_group,is_long_contract,churn,gender,contract_type
0,C001,34,59000,120,1,female,monthly,2025-01-10,2026-06-01,2025,...,0,0,1,0,0,middle,0,0,female,month-to-month
1,C002,45,79000,380,3,male,one_year,2024-12-01,2026-05-10,2024,...,0,0,1,0,1,long,1,1,male,1-year
2,C003,41,72000,45,0,female,monthly,2026-04-20,2026-06-03,2026,...,0,0,1,0,0,middle,0,0,female,month-to-month
3,C004,52,99000,800,5,male,one_year,2023-05-15,2026-03-01,2023,...,0,0,1,0,1,long,1,1,male,1-year
4,C005,41,69000,220,2,female,unknown,2025-08-03,2026-05-30,2025,...,0,0,1,0,0,middle,0,0,female,unknown


In [176]:
df_parquet_check.dtypes

customer_id                             str
age                                   int64
monthly_fee                           int64
usage_days                            int64
support_calls                         int64
gender_standard                         str
contract_type_standard                  str
signup_date                  datetime64[ns]
last_login_date              datetime64[ns]
signup_year                           int32
signup_month                          int32
last_login_dayofweek                  int32
last_login_day_name                     str
days_since_signup                     int64
days_since_last_login                 int64
age_was_missing                       int64
monthly_fee_was_missing               int64
gender_was_missing                    int64
contract_type_was_missing             int64
monthly_fee_outlier                   int64
support_calls_outlier                 int64
is_high_fee                           int64
is_short_user                   

In [177]:
format_compare = pd.DataFrame({
    "format": ["csv", "parquet"],
    "row_count": [df_csv_check.shape[0], df_parquet_check.shape[0]],
    "column_count": [df_csv_check.shape[1], df_parquet_check.shape[1]]
})
format_compare

,format,row_count,column_count
0,csv,9,29
1,parquet,9,29


In [178]:
dtype_compare = pd.DataFrame({
    "csv_dtype": df_csv_check.dtypes.astype(str),
    "parquet_dtype": df_parquet_check.dtypes.astype(str)
})
dtype_compare

,csv_dtype,parquet_dtype
customer_id,str,str
age,int64,int64
monthly_fee,int64,int64
usage_days,int64,int64
support_calls,int64,int64
gender_standard,str,str
contract_type_standard,str,str
signup_date,str,datetime64[ns]
last_login_date,str,datetime64[ns]
signup_year,int64,int32


In [179]:
clean_v2_summary_table = pd.DataFrame({
    "item": [
        "row_count",
        "column_count",
        "missing_total",
        "full_duplicate_count",
        "customer_id_duplicate_count",
        "csv_path",
        "parquet_path"
    ],
    "value": [
        df_clean_v2.shape[0],
        df_clean_v2.shape[1],
        int(df_clean_v2.isnull().sum().sum()),
        int(df_clean_v2.duplicated().sum()),
        int(df_clean_v2.duplicated(subset=["customer_id"]).sum()),
        str(clean_v2_csv),
        str(clean_v2_parquet)
    ]
})
clean_v2_summary_table


,item,value
0,row_count,9
1,column_count,29
2,missing_total,0
3,full_duplicate_count,0
4,customer_id_duplicate_count,0
5,csv_path,/home/soldesk/ai-hybrid-lab-kim-juil/data/clea...
6,parquet_path,/home/soldesk/ai-hybrid-lab-kim-juil/data/clea...


In [180]:
import json
from pathlib import Path
day2_final_summary = {
    "author": "student",
    "input_raw_path": str(raw_file) if "raw_file" in globals() else "data/raw/customers_raw.csv",
    "working_dataframe": "df_clean_v2",
    "row_count": int(df_clean_v2.shape[0]),
    "column_count": int(df_clean_v2.shape[1]),
    "missing_total": int(df_clean_v2.isnull().sum().sum()),
    "full_duplicate_count": int(df_clean_v2.duplicated().sum()),
    "customer_id_duplicate_count": int(df_clean_v2.duplicated(subset=["customer_id"]).sum()),
    "categorical_validation": {
        "gender_invalid_values": sorted(list(gender_invalid_final)),
        "contract_type_invalid_values": sorted(list(contract_invalid_final)),
        "usage_group_invalid_values": sorted(list(usage_group_invalid_final))
    },
    "date_validation": date_final_summary,
    "created_features": [
        "age_was_missing",
        "monthly_fee_was_missing",
        "gender_was_missing",
        "contract_type_was_missing",
        "signup_year",
        "signup_month",
        "last_login_dayofweek",
        "last_login_day_name",
        "days_since_signup",
        "days_since_last_login",
        "monthly_fee_outlier",
        "support_calls_outlier",
        "is_high_fee",
        "is_short_user",
        "has_many_support_calls",
        "usage_group",
        "is_long_contract"
    ],
    "saved_outputs": {
        "clean_csv": str(clean_v2_csv),
        "clean_parquet": str(clean_v2_parquet),
        "preprocessing_report": "outputs/week2/preprocessing_report.md"
    },
    "g4dn_used": False,
    "openshift_deploy_used": False,
    "h200_used": False,
    "note": "Day 2 8교시는 최종 clean v2 데이터를 CSV/Parquet으로 저장하고 전처리 리포트를 작성한 시간"
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
day2_final_summary_path = OUTPUT_DIR / "day2_final_preprocessing_summary.json"
with open(day2_final_summary_path, "w", encoding="utf-8") as f:
    json.dump(day2_final_summary, f, ensure_ascii=False, indent=2)
print("saved:", day2_final_summary_path)
day2_final_summary


saved: /home/soldesk/ai-hybrid-lab-kim-juil/outputs/week2/day2_final_preprocessing_summary.json


{'author': 'student',
 'input_raw_path': '/home/soldesk/ai-hybrid-lab-kim-juil/data/raw/customers_raw.csv',
 'working_dataframe': 'df_clean_v2',
 'row_count': 9,
 'column_count': 29,
 'missing_total': 0,
 'full_duplicate_count': 0,
 'customer_id_duplicate_count': 0,
 'categorical_validation': {'gender_invalid_values': [],
  'contract_type_invalid_values': [],
  'usage_group_invalid_values': []},
 'date_validation': {'signup_date_dtype': 'datetime64[ns]',
  'last_login_date_dtype': 'datetime64[ns]',
  'signup_date_nat_count': 0,
  'last_login_date_nat_count': 0,
  'negative_days_since_signup_count': 0,
  'negative_days_since_last_login_count': 0},
 'created_features': ['age_was_missing',
  'monthly_fee_was_missing',
  'gender_was_missing',
  'contract_type_was_missing',
  'signup_year',
  'signup_month',
  'last_login_dayofweek',
  'last_login_day_name',
  'days_since_signup',
  'days_since_last_login',
  'monthly_fee_outlier',
  'support_calls_outlier',
  'is_high_fee',
  'is_short_use